<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_42_ai_dev_tools/note_lesson_42_ai_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Урок 42 — AI-інструменти розробника: ризики й валідація

| Крок | Що робимо |
|---|---|
| 0 | `news_hub` з уроку 42 і `depression_dashboard` — без ключа AI: вправи на справжньому результаті агента |
| 1 | hook і дозволи: як проєкт обмежує агента (вправи 1–2) |
| 2 | запуск A: код, який агент перевірив лише своїми тестами (вправи 3–4) |
| 3 | запуск B і рецензія: що специфікація не питала (вправи 5–6) |
| 4 | `depression_dashboard`: правдоподібна неправда (вправи 7–10) |

Розбір експерименту — на [сторінці уроку](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_42/); довідник — [Claude Code](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/ai/claude_code/).

---
## 0. Проєкти

Клітинка завантажує урок (у Colab), ставить залежності й переходить у папку `news_hub`. Пакет `depression_dashboard` підключаємо за шляхом.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_42_ai_dev_tools"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import fakeredis, fastapi, flask, pytest, sklearn, tzdata  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt",
                    "flask>=3.0.3", "pandas>=2.2.3", "scikit-learn>=1.5.2"], check=True)

LESSON_DIR = os.getcwd()
DASHBOARD = os.path.join(LESSON_DIR, "depression_dashboard")
os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
sys.path.append(DASHBOARD)
os.environ["REDIS_URL"] = "fakeredis://"
print(open("CLAUDE.md", encoding="utf-8").read()[:600])

---
## 1. Як проєкт обмежує агента

### Вправа 1

Hook `.claude/hooks/unit_tests.py` Claude Code запускає після кожної правки. На stdin він отримує JSON події. Запусти hook сам — як це робить Claude Code — для двох подій: правка `news_hub/rss.py` і правка `README.md`. Коди виходу збережи в `code_py` і `code_md`.

In [ ]:
import json


def run_hook(file_path: str) -> subprocess.CompletedProcess:
    event = {"hook_event_name": "PostToolUse", "tool_name": "Edit", "tool_input": {"file_path": file_path}}
    return subprocess.run([sys.executable, ".claude/hooks/unit_tests.py"], input=json.dumps(event),
                          capture_output=True, text=True)


# YOUR CODE HERE
# BEGIN SOLUTION
code_py = run_hook("news_hub/rss.py").returncode
code_md = run_hook("README.md").returncode
# END SOLUTION

print("rss.py →", code_py, "| README.md →", code_md)
assert (code_py, code_md) == (0, 0)
print("✅ Вправа 1 пройдена — тести зелені; для не-.py файлу hook тести й не запускав")

**🔮 Прогноз:** Агент зламав `news_hub/rss.py` так, що `pytest -m unit` падає. Який код виходу поверне hook і що побачить агент?

<details>
<summary>Відповідь</summary>

`2` і хвіст виводу pytest у stderr. Для `PostToolUse` код 2 не скасовує правку (вона вже відбулась), але Claude Code передає stderr агентові як помилку, яку треба виправити, перш ніж іти далі.
</details>

### Вправа 2

Дозволи з `.claude/settings.json`. Напиши спрощений `decide(command)` для правил виду `Bash(префікс *)`:

- команду розбиваємо за `&&`, `||`, `;`, `|` — **кожна** частина перевіряється окремо;
- частина потрапляє під правило, якщо дорівнює префіксу або починається з «префікс + пробіл»;
- порядок: хоч одна частина під `deny` → `"deny"`; хоч одна під `ask` або не під жодним правилом → `"ask"`; інакше `"allow"`.

In [ ]:
import re

settings = json.load(open(".claude/settings.json", encoding="utf-8"))["permissions"]


def bash_prefixes(rules: list[str]) -> list[str]:
    return [m.group(1) for rule in rules if (m := re.fullmatch(r"Bash\((.+?)(?: \*)?\)", rule))]


ALLOW, ASK, DENY = (bash_prefixes(settings.get(key, [])) for key in ("allow", "ask", "deny"))


def covered(part: str, prefixes: list[str]) -> bool:
    return any(part == p or part.startswith(p + " ") for p in prefixes)


def decide(command: str) -> str:
    parts = [p.strip() for p in re.split(r"&&|\|\||;|\|", command) if p.strip()]
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if any(covered(p, DENY) for p in parts):
        return "deny"
    if all(covered(p, ALLOW) for p in parts):
        return "allow"
    return "ask"
    # END SOLUTION


cases = {"pytest -m unit": "allow", "mypy --strict news_hub": "allow", "git push origin main": "deny",
         "pip install requests": "deny", "pytest && rm -rf .": "ask", "git status && git push": "deny"}
for command, expected in cases.items():
    print(f"{command:28} → {decide(command)}")
    assert decide(command) == expected, command
print("✅ Вправа 2 пройдена")

Справжній Claude Code перевіряє складніше (обгортки `timeout`, підоболонки, змінні), але принцип той самий: `deny` сильніший за `allow`, і складена команда не «прослизне» за префіксом першої частини. А `allow` з файлу репозиторію діє лише після того, як ти **довірив** папку.

---
## 2. Запуск A: зелений на власних тестах

Агент A не бачив стрічки й написав фікстуру сам — лише з датами `+0300`. Ось його функція часу, дослівно:

In [ ]:
from email.utils import parsedate_to_datetime


def pravda_time_run_a(pub_date: str) -> str:
    """RFC 822 (`<pubDate>`) → «HH:MM:SS», як час у стрічці rbc.ua; не розібрали — порожній рядок."""
    if not pub_date:
        return ""
    try:
        return parsedate_to_datetime(pub_date).strftime("%H:%M:%S")
    except (TypeError, ValueError):
        return ""


print(pravda_time_run_a("Sat, 26 Sep 2026 21:40:00 +0300"))

### Вправа 3

Знайди вхід, на якому `pravda_time_run_a` помиляється. Збережи в `bad_pub_date` рядок `pubDate`, для якого його відповідь не збігається з київським часом від фінального `news_hub.rss._kyiv_datetime` (візьми з нього лише частину після `T`).

In [ ]:
from news_hub.rss import _kyiv_datetime

# YOUR CODE HERE
# BEGIN SOLUTION
bad_pub_date = "Sat, 26 Sep 2026 17:05:00 GMT"
# END SOLUTION

print("запуск A:", pravda_time_run_a(bad_pub_date), "| Київ:", _kyiv_datetime(bad_pub_date).split("T")[1])
assert pravda_time_run_a(bad_pub_date) != _kyiv_datetime(bad_pub_date).split("T")[1]
print("✅ Вправа 3 пройдена — час у GMT агент A записував як київський")

### Вправа 4

Тест із запуску A: `assert parse_pravda_rss("не xml") == []`. Він закріплює «тиху» відмову. Напиши перевірку-вимогу `raises_on_garbage(parse)`: `True`, якщо `parse` на битому XML кидає `ValueError`. Перевір її на двох версіях: `parse_run_a` (поведінка агента A) і фінальній `parse_pravda_rss`.

In [ ]:
import xml.etree.ElementTree as ET

from news_hub.rss import parse_pravda_rss


def parse_run_a(xml: str) -> list:
    try:
        root = ET.fromstring(xml)
    except ET.ParseError:
        return []
    return [item for item in root.iter("item")]


# YOUR CODE HERE
# BEGIN SOLUTION
def raises_on_garbage(parse) -> bool:
    try:
        parse("<rss><channel><item><title>обрізано")
    except ValueError:
        return True
    return False
# END SOLUTION

print("запуск A:", raises_on_garbage(parse_run_a), "| фінальний:", raises_on_garbage(parse_pravda_rss))
assert (raises_on_garbage(parse_run_a), raises_on_garbage(parse_pravda_rss)) == (False, True)
print("✅ Вправа 4 пройдена")

---
## 3. Запуск B і рецензія

Агент B отримав специфікацію `tests/unit/test_pravda.py` і зробив її зеленою. Рецензія знайшла три випадки, про які специфікація мовчала (`tests/unit/test_pravda_review.py`).

### Вправа 5

Запусти обидва файли тестів через pytest і збережи останній рядок виводу в `summary`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no",
                         "tests/unit/test_pravda.py", "tests/unit/test_pravda_review.py"],
                        capture_output=True, text=True)
summary = result.stdout.strip().splitlines()[-1]
# END SOLUTION

print(summary)
assert summary.startswith("21 passed")
print("✅ Вправа 5 пройдена")

**🔮 Прогноз:** Сайт віддав замість RSS сторінку помилки `<html><body><h1>403</h1></body></html>` (правильний XML, код 200). Що поверне `parse_pravda_rss`?

<details>
<summary>Відповідь</summary>

`[]` — так само, як на порожній стрічці. Збір «успішний», новин 0, помилки ніхто не побачить. Це розібраний приклад на сторінці уроку.
</details>

### Вправа 6

Напиши `parse_strict(xml)`: обгортка над `parse_pravda_rss`, яка кидає `ValueError`, якщо корінь документа — не `<rss>` з `<channel>`. Порожня стрічка `<rss><channel></channel></rss>` — легальна: `[]`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
def parse_strict(xml: str) -> list:
    root = ET.fromstring(xml)
    if root.tag != "rss" or root.find("channel") is None:
        raise ValueError(f"очікуємо RSS (<rss><channel>), а отримали <{root.tag}>")
    return parse_pravda_rss(xml)
# END SOLUTION

assert parse_strict("<rss><channel></channel></rss>") == []
try:
    parse_strict("<html><body><h1>403 Forbidden</h1></body></html>")
    raise AssertionError("HTML мав дати ValueError")
except ValueError as error:
    print("ValueError:", error)
feed = open("tests/fixtures/pravda_rss.xml", encoding="utf-8").read()
assert len(parse_strict(feed)) == 4
print("✅ Вправа 6 пройдена")

---
## 4. `depression_dashboard`: правдоподібна неправда

Проєкт, який AI згенерував у старому курсі. Бекенд — Flask; дані — синтетичні, з колонками справжнього датасету (генератор з `tests/conftest.py` проєкту).

In [ ]:
import importlib.util
import tempfile
from pathlib import Path

spec = importlib.util.spec_from_file_location("dd_conftest", os.path.join(DASHBOARD, "tests", "conftest.py"))
dd_conftest = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dd_conftest)

data_dir = Path(tempfile.mkdtemp())
dd_conftest.make_students(600).to_csv(data_dir / "Student Depression Dataset.csv", index=False)
dd_conftest.make_burnout().to_csv(data_dir / "mental_health_burnout_tech_2026.csv", index=False)

from backend.config import config
from backend.services import data_service, ml_service

config.STUDENT_DATASET = data_dir / "Student Depression Dataset.csv"
config.BURNOUT_DATASET = data_dir / "mental_health_burnout_tech_2026.csv"
ml_service.MODEL_PATH = data_dir / "models" / "classifier.pkl"
data_service.load_student_df.cache_clear()

from backend.app import create_app

dd = create_app().test_client()
print(dd.get("/health").get_json(), "| рядків:", len(data_service.load_student_df()))

### Вправа 7

Старий код віддавав важливість ознак через стандартний `jsonify`. Збери `Flask`-застосунок **без** виправлення (`Flask("old")`) і поверни через `jsonify` словник `importance` у тому порядку, який отримав би клієнт. Список ключів — у `old_order`.

In [ ]:
from flask import Flask, jsonify

importance = dd.get("/api/feature-importance").get_json()          # виправлений бекенд: за спаданням
print("виправлений бекенд:", list(importance)[:4])

# YOUR CODE HERE
# BEGIN SOLUTION
old_app = Flask("old")
with old_app.app_context():
    old_order = list(json.loads(jsonify(importance).get_data(as_text=True)))
# END SOLUTION

print("старий jsonify:    ", old_order[:4])
assert old_order == sorted(importance) and old_order != list(importance)
print("✅ Вправа 7 пройдена — «топ-12» на графіку були першими 12 за абеткою")

### Вправа 8

`POST /api/predict` тепер приймає лише анкету (`StudentProfile`). Надішли анкету з полем `"CGPA": 7.5` (повзунок старого UI, якого немає в ознаках моделі) і збережи статус у `status` і поля помилок у `fields`.

In [ ]:
PROFILE = {"age": 22, "gender": "Female", "sleep_duration": "5-6 hours", "academic_pressure": 3,
           "work_pressure": 0, "study_satisfaction": 3, "job_satisfaction": 0, "work_study_hours": 6,
           "financial_stress": 2, "dietary_habits": "Moderate", "family_history": False, "suicidal_thoughts": False}
print(dd.post("/api/predict", json=PROFILE).get_json())

# YOUR CODE HERE
# BEGIN SOLUTION
response = dd.post("/api/predict", json={**PROFILE, "CGPA": 7.5})
status = response.status_code
fields = [e["field"] for e in response.get_json()["errors"]]
# END SOLUTION

print(status, fields)
assert (status, fields) == (422, ["CGPA"])
print("✅ Вправа 8 пройдена — поле, якого модель не знає, тепер помилка, а не тиша")

### Вправа 9

Старий UI називав кластери словником у коді: `{0: "High Pressure", 1: "Sleep Deprived", 2: "Stable / Low Risk", 3: "Financially Stressed"}`. Візьми з `/api/clusters` профілі й знайди номер кластера, що **найменше** спить (`Sleep_hours`). Порівняй зі старою назвою.

In [ ]:
OLD_LABELS = {0: "High Pressure", 1: "Sleep Deprived", 2: "Stable / Low Risk", 3: "Financially Stressed"}
clusters = dd.get("/api/clusters").get_json()

# YOUR CODE HERE
# BEGIN SOLUTION
least_sleep = min(clusters["cluster_profiles"], key=lambda cid: clusters["cluster_profiles"][cid]["Sleep_hours"])
# END SOLUTION

print("найменше сну — кластер", least_sleep, "| стара назва:", OLD_LABELS[int(least_sleep)],
      "| нова:", clusters["cluster_names"][least_sleep])
assert clusters["cluster_profiles"][least_sleep]["Sleep_hours"] == min(
    p["Sleep_hours"] for p in clusters["cluster_profiles"].values())
print("✅ Вправа 9 пройдена — номери KMeans довільні, тож вшита назва вгадує лише випадково")

---
## Знайди помилку

### Вправа 10

У старому `get_depression_by_group()` кешований DataFrame змінювали на місці. Ось той самий патерн у мініатюрі. Виправ `groups()`, щоб кеш лишався незмінним.

In [ ]:
from functools import lru_cache

import pandas as pd


@lru_cache(maxsize=1)
def load() -> pd.DataFrame:
    return pd.DataFrame({"stress": [1, 3, 5], "depression": [0, 1, 1]})


def groups() -> dict:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    df = load().copy()
    # END SOLUTION
    df["bucket"] = pd.cut(df["stress"], bins=[0, 2, 5], labels=["low", "high"])
    return df.groupby("bucket", observed=True)["depression"].mean().to_dict()


print(groups())
print("колонки кешу:", list(load().columns))
assert list(load().columns) == ["stress", "depression"]
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Чим `CLAUDE.md` відрізняється від `permissions.deny`?
2. Чому 101 зелений тест запуску A не доводить правильності коду?
3. Що дала специфікація запуску B і чого не дала?
4. Чому вада з `jsonify` не видна на скриншоті дашборду?
5. Три пункти з чекліста рецензії AI-коду, які ти застосуєш у своєму проєкті першими.

## Далі

- Книга курсу: [Урок 42](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_42/) — експеримент з двома запусками, рецензія, десять вад дашборду, чекліст.
- Спробуй сам: встанови Claude Code ([довідник](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/ai/claude_code/)), відкрий `news_hub` і виконай `/add-news-source` для третього джерела.
- Урок 43 — Gemini в агрегаторі: відповідь LLM — теж дані, які перевіряє Pydantic.